Clean Silver - Live Production Data
Reads bronze battery data, applies cleaning/filtering, saves to silver.

**Input**: bronze/live/battery_full_history.json
**Output**: silver/erp/battery/battery_clean_live.json

In [0]:
%run ./_local_config

In [0]:
import sys
sys.path.append("/Workspace/Users/venura-it@brownsgroup.com/Exide sales/Exide-Sales-Forecast")

from src.io.storage import get_blob_service, read_bronze, save_silver
from src.transform.clean_silver import clean_to_silver
import pandas as pd
import datetime

blob_service = get_blob_service(storage_account_name, storage_account_key)

In [0]:
import src.io.storage as storage_module
print(dir(storage_module))

In [0]:
bronze = read_bronze(blob_service, "live/battery_full_history.json")
bronze["postingDate"] = pd.to_datetime(bronze["postingDate"])

today = pd.Timestamp(datetime.date.today())
bronze_trimmed = bronze[bronze["postingDate"] < today].copy()

print(f"Bronze: {len(bronze)} rows -> trimmed: {len(bronze_trimmed)} rows")
print(f"Date range: {bronze_trimmed['postingDate'].min()} to {bronze_trimmed['postingDate'].max()}")

In [0]:
silver = clean_to_silver(bronze_trimmed)
print(f"Silver: {silver.shape}")
print(f"Date range: {silver['posting_date'].min()} to {silver['posting_date'].max()}")

save_silver(blob_service, silver, "live/battery/battery_clean_live.json")
print("Saved to silver/live/battery/battery_clean_live.json")